# Hybrid partial-BZ mixture SCM

Новая версия: сначала учим стабильные $A,B,p_e(Z)$, затем учим режимные $C_e$ из задачи

$$Y - \rho B\mathbb{E}[Z] \approx C_e X,$$

а предсказание строим как

$$\hat Y = \sum_e r_e\left(\rho B\mathbb{E}[Z\mid X,e] + C_e X\right).$$

Режимы по умолчанию выделяются по intensity-features для временных рядов, но сама модель остается модульной: для text-to-text вместо них можно подставить embedding features.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.linear_model import Ridge

from causal_hybrid_partial_bz_scm import (
    synchronize_phone_zips, selected_signal_columns, intensity_regime_columns,
    select_lag_by_cca, make_lagged_dataset, mask_from_intervals,
    prediction_metrics, CCALatentProjector, HybridPartialBZConfig,
    HybridPartialBZMixtureSCM, fit_predict_baselines, top_operator_edges,
    c_difference_norm, c_singular_values,
)

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})


## 1. Конфигурация

In [ ]:
HAND_ZIP = Path("hand.zip")
BELT_ZIP = Path("belt.zip")

USE_ACCELEROMETER = True
USE_GYROSCOPE = True
USE_ORIENTATION = False

LATENT_DIM = 3
N_REGIMES = 2
N_LAGS = 12
WINDOW_TAU = 1
HORIZON = 1

# Валидируемся строго внутри двух заданных физических режимов.
VALIDATION_INTERVALS = [(5.0, 6.0), (12.0, 13.0)]
VALIDATION_NAMES = ["elbow_to_belt_5_6s", "wrist_to_belt_12_13s"]

# Causal hybrid mode: Y - rho BZ ~= C_e X
BZ_WEIGHT = 0.5
RIDGE_A = 1e-2
RIDGE_B = 1e-2
RIDGE_C = 1e-2
SIGMA_Y_FLOOR = 5e-2
C_NORM_CLIP = None  # например 10.0, если C начнет взрываться

MAX_ITER_AB = 80
RANDOM_STATE = 42


## 2. Загрузка, синхронизация и первичная визуализация

In [ ]:
synced, sync_info = synchronize_phone_zips(
    HAND_ZIP, BELT_ZIP,
    use_accelerometer=USE_ACCELEROMETER,
    use_gyroscope=USE_GYROSCOPE,
    use_orientation=USE_ORIENTATION,
)
print(sync_info)
synced.head()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
t = synced["t_rel"].to_numpy()
for sensor in ["acc", "gyro"]:
    c = f"hand_{sensor}_dyn_norm_smooth"
    if c in synced:
        axes[0].plot(t, synced[c], label=c)
for sensor in ["acc", "gyro"]:
    c = f"belt_{sensor}_dyn_norm_smooth"
    if c in synced:
        axes[1].plot(t, synced[c], label=c)
for ax in axes:
    for a,b in VALIDATION_INTERVALS:
        ax.axvspan(a,b, alpha=0.12)
    ax.legend(loc="upper right")
axes[0].set_title("Hand movement intensity")
axes[1].set_title("Belt movement intensity")
axes[1].set_xlabel("time, sec")
plt.tight_layout()


## 3. Выбор лага и построение supervised dataset

In [ ]:
x_cols = selected_signal_columns(
    synced, "hand",
    use_accelerometer=USE_ACCELEROMETER,
    use_gyroscope=USE_GYROSCOPE,
    use_orientation=USE_ORIENTATION,
)
y_cols = selected_signal_columns(
    synced, "belt",
    use_accelerometer=USE_ACCELEROMETER,
    use_gyroscope=USE_GYROSCOPE,
    use_orientation=USE_ORIENTATION,
)

# Для выбора лага используем динамические нормы, если они есть; иначе основные каналы.
lag_x_cols = [c for c in ["hand_acc_dyn_norm_smooth", "hand_gyro_dyn_norm_smooth"] if c in synced.columns]
lag_y_cols = [c for c in ["belt_acc_dyn_norm_smooth", "belt_gyro_dyn_norm_smooth"] if c in synced.columns]
if not lag_x_cols or not lag_y_cols:
    lag_x_cols, lag_y_cols = x_cols, y_cols

max_lag = int(min(80, max(5, sync_info["target_hz"] * 0.8)))
best_lag, lag_table = select_lag_by_cca(synced, lag_x_cols, lag_y_cols, min_lag=1, max_lag=max_lag)
print("best_lag:", best_lag, "seconds:", best_lag / sync_info["target_hz"])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(lag_table["lag"] / sync_info["target_hz"], lag_table["cca_corr"], marker="o")
ax.axvline(best_lag / sync_info["target_hz"], linestyle="--", label=f"best lag = {best_lag}")
ax.set_xlabel("lag, sec")
ax.set_ylabel("1D CCA correlation")
ax.set_title("Lag selection by dynamic features")
ax.legend()
plt.tight_layout()


In [ ]:
regime_extra_cols = intensity_regime_columns(synced)
print("X columns:", x_cols)
print("Y columns:", y_cols)
print("Regime/intensity columns:", regime_extra_cols)

ds = make_lagged_dataset(
    synced, x_cols, y_cols,
    causal_lag=best_lag, n_lags=N_LAGS, window_tau=WINDOW_TAU, horizon=HORIZON,
    extra_regime_cols=regime_extra_cols,
)
X, Y, t_ds = ds["X"], ds["Y"], ds["t"]
regime_extra = ds["regime_extra"]
print(X.shape, Y.shape, regime_extra.shape, t_ds.min(), t_ds.max())

val_mask = mask_from_intervals(t_ds, VALIDATION_INTERVALS)
train_mask = ~val_mask
print("train:", train_mask.sum(), "validation:", val_mask.sum())


## 4. CCA-латент и обучение hybrid partial-BZ SCM

In [ ]:
projector = CCALatentProjector(n_components=LATENT_DIM).fit(X[train_mask], Y[train_mask])
X_lat_all, Y_lat_all = projector.transform(X, Y)

# Для режимов используем intensity GMM; для text-to-text сюда можно подать embedding-level regime features.
regime_features_all = regime_extra

cfg = HybridPartialBZConfig(
    n_regimes=N_REGIMES,
    latent_dim=LATENT_DIM,
    max_iter_ab=MAX_ITER_AB,
    ridge_a=RIDGE_A,
    ridge_b=RIDGE_B,
    ridge_c=RIDGE_C,
    sigma_y_floor=SIGMA_Y_FLOOR,
    c_mode="partial_bz_direct",
    bz_weight=BZ_WEIGHT,
    c_norm_clip=C_NORM_CLIP,
    random_state=RANDOM_STATE,
)
model = HybridPartialBZMixtureSCM(cfg).fit(
    X_lat_all[train_mask],
    Y_lat_all[train_mask],
    regime_features_all[train_mask],
)

hist = model.history_dataframe()
print("n_iter:", model.n_iter_done_)
print("A norm:", np.linalg.norm(model.A_))
print("B norm:", np.linalg.norm(model.B_))
print("C norms:", model.direct_channel_norms())
print("||C0-C1||_F:", c_difference_norm(model))
print("sigma_x, sigma_y:", model.sigma_x_, model.sigma_y_)


## 5. Графики обучения Stage-1

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
axes[0,0].plot(hist["iter"], hist["objective"], marker="o")
axes[0,0].set_title("Stage-1 objective")
axes[0,0].set_xlabel("iteration")
axes[0,0].set_ylabel("objective")

axes[0,1].plot(hist["iter"], hist["x_mse"], marker="o", label="X MSE")
axes[0,1].plot(hist["iter"], hist["y_mse"], marker="o", label="Y MSE")
axes[0,1].set_title("Reconstruction MSE")
axes[0,1].legend()

axes[1,0].plot(hist["iter"], hist["A_norm"], label="||A||")
axes[1,0].plot(hist["iter"], hist["B_norm"], label="||B||")
axes[1,0].set_title("A/B norms")
axes[1,0].legend()

axes[1,1].plot(hist["iter"], hist["sigma_x"], label="sigma_x")
axes[1,1].plot(hist["iter"], hist["sigma_y"], label="sigma_y")
axes[1,1].axhline(SIGMA_Y_FLOOR, linestyle="--", label="sigma_y floor")
axes[1,1].set_title("Noise scales")
axes[1,1].legend()
plt.tight_layout()


## 6. Responsibilities и режимы

In [ ]:
resp_all = model.regime_responsibilities(regime_features_all)
hard_all = resp_all.argmax(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for e in range(N_REGIMES):
    axes[0].plot(t_ds, resp_all[:, e], label=f"regime {e}")
axes[0].set_title("GMM responsibilities by intensity regime features")
axes[0].set_ylabel("responsibility")
axes[0].legend()

axes[1].step(t_ds, hard_all, where="mid")
axes[1].set_title("Hard regime labels")
axes[1].set_ylabel("regime")
axes[1].set_xlabel("time, sec")
for ax in axes:
    for a,b in VALIDATION_INTERVALS:
        ax.axvspan(a,b, alpha=0.12)
plt.tight_layout()

print("regime counts:", pd.Series(hard_all).value_counts().sort_index().to_dict())
print("switches:", int(np.sum(hard_all[1:] != hard_all[:-1])))


## 7. Предсказание и baseline

In [ ]:
Y_lat_pred, pred_details = model.predict_latent(X_lat_all, regime_features_all, return_details=True)
Y_pred = projector.inverse_y(Y_lat_pred)

# Baselines.
ridge_raw = Ridge(alpha=1.0).fit(X[train_mask], Y[train_mask])
Y_pred_raw = ridge_raw.predict(X)

ridge_cca = Ridge(alpha=1.0).fit(X_lat_all[train_mask], Y_lat_all[train_mask])
Y_pred_cca = projector.inverse_y(ridge_cca.predict(X_lat_all))

rows = []
for name, pred in [
    ("Hybrid partial-BZ SCM", Y_pred),
    ("Raw Ridge X->Y", Y_pred_raw),
    ("CCA latent + Ridge", Y_pred_cca),
]:
    m = prediction_metrics(Y[val_mask], pred[val_mask])
    rows.append({"model": name, **m})
metrics_df = pd.DataFrame(rows)
metrics_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
metrics_df.plot.bar(x="model", y="NMSE", ax=axes[0], legend=False)
axes[0].set_title("Validation NMSE")
axes[0].tick_params(axis="x", rotation=20)
metrics_df.plot.bar(x="model", y="R2", ax=axes[1], legend=False)
axes[1].set_title("Validation R²")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()


## 8. Метрики по двум ручным validation-интервалам

In [ ]:
interval_rows = []
for name, (a,b) in zip(VALIDATION_NAMES, VALIDATION_INTERVALS):
    msk = (t_ds >= a) & (t_ds <= b)
    for model_name, pred in [
        ("Hybrid partial-BZ SCM", Y_pred),
        ("Raw Ridge X->Y", Y_pred_raw),
        ("CCA latent + Ridge", Y_pred_cca),
    ]:
        if msk.sum() > 2:
            interval_rows.append({"interval": name, "model": model_name, **prediction_metrics(Y[msk], pred[msk])})
interval_metrics_df = pd.DataFrame(interval_rows)
interval_metrics_df


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for model_name, sub in interval_metrics_df.groupby("model"):
    ax.plot(sub["interval"], sub["NMSE"], marker="o", label=model_name)
ax.set_title("NMSE on fixed validation intervals")
ax.set_ylabel("NMSE")
ax.legend()
plt.tight_layout()


## 9. Диагностика $C_e$: сингулярные числа и отличие режимов

In [ ]:
sv_df = c_singular_values(model)
print("||C0-C1||_F:", c_difference_norm(model))
print("C norms:", model.direct_channel_norms())
sv_df.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for e, sub in sv_df.groupby("regime"):
    ax.plot(sub["index"], sub["singular_value"], marker="o", label=f"C regime {e}")
ax.set_title("Singular values of direct channels C_e")
ax.set_xlabel("index")
ax.set_ylabel("singular value")
ax.legend()
plt.tight_layout()

if N_REGIMES >= 2:
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(model.C_[0] - model.C_[1], aspect="auto")
    ax.set_title("C_0 - C_1 heatmap")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()


## 10. Графы причинности по режимам

In [ ]:
def draw_operator_graph(op, x_names, y_names, title, top_k=15):
    edges = top_operator_edges(op, x_names, y_names, top_k=top_k)
    G = nx.DiGraph()
    for _, row in edges.iterrows():
        G.add_edge(row["source"], row["target"], weight=row["weight"], abs_weight=row["abs_weight"])
    pos = {}
    srcs = list(dict.fromkeys(edges["source"].tolist()))
    tgts = list(dict.fromkeys(edges["target"].tolist()))
    for i, n in enumerate(srcs):
        pos[n] = (0, -i)
    for i, n in enumerate(tgts):
        pos[n] = (1, -i)
    fig, ax = plt.subplots(figsize=(12, max(5, 0.35*(len(srcs)+len(tgts)))))
    widths = [1 + 4*G[u][v]["abs_weight"]/max(edges["abs_weight"].max(), 1e-12) for u,v in G.edges()]
    nx.draw_networkx_nodes(G, pos, nodelist=srcs, node_shape="s", node_size=800, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=tgts, node_shape="o", node_size=800, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    nx.draw_networkx_edges(G, pos, width=widths, arrows=True, arrowsize=14, alpha=0.65, ax=ax)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    return edges

# В latent-графе имена компонент сингулярного канала/CCA latent.
x_lat_names = [f"X_lat_{i}" for i in range(LATENT_DIM)]
y_lat_names = [f"Y_lat_{i}" for i in range(LATENT_DIM)]
all_edges=[]
for e in range(N_REGIMES):
    edges = draw_operator_graph(model.C_[e], x_lat_names, y_lat_names, f"Direct latent graph C_{e}", top_k=12)
    edges["regime"] = e
    all_edges.append(edges)
edges_df = pd.concat(all_edges, ignore_index=True)
edges_df


## 11. Сохранение результатов

In [ ]:
metrics_df.to_csv("hybrid_partial_bz_metrics.csv", index=False)
interval_metrics_df.to_csv("hybrid_partial_bz_interval_metrics.csv", index=False)
sv_df.to_csv("hybrid_partial_bz_c_singular_values.csv", index=False)
edges_df.to_csv("hybrid_partial_bz_c_edges.csv", index=False)
hist.to_csv("hybrid_partial_bz_training_history.csv", index=False)
print("saved csv outputs")
